In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import sys
import io
import warnings
warnings.filterwarnings('ignore')

In [ ]:
DATA_DIR = "orgdata/"
OUT_DIR = "cleandata/"
def load(name, **kw):
    path = os.path.join(DATA_DIR, name)
    df = pd.read_csv(path, **kw)
    return df

def save(df, name):
    path = os.path.join(OUT_DIR, name)
    df.to_csv(path, index=False)

In [3]:
orders      = load('orders.csv',      parse_dates=['order_date'])
order_items = load('order_items.csv')
payments    = load('payments.csv')
products    = load('products.csv')
customers   = load('customers.csv',   parse_dates=['signup_date'])
geography   = load('geography.csv')
promotions  = load('promotions.csv',  parse_dates=['start_date', 'end_date'])
shipments   = load('shipments.csv',   parse_dates=['ship_date', 'delivery_date'])
returns     = load('returns.csv',     parse_dates=['return_date'])
reviews     = load('reviews.csv',     parse_dates=['review_date'])
inventory   = load('inventory.csv',   parse_dates=['snapshot_date'])
web_traffic = load('web_traffic.csv', parse_dates=['date'])
sales       = load('sales.csv',      parse_dates=['Date'])

  ✓ Loaded orders.csv: 646,945 rows × 8 cols
  ✓ Loaded order_items.csv: 714,669 rows × 7 cols
  ✓ Loaded payments.csv: 646,945 rows × 4 cols
  ✓ Loaded products.csv: 2,412 rows × 8 cols
  ✓ Loaded customers.csv: 121,930 rows × 7 cols
  ✓ Loaded geography.csv: 39,948 rows × 4 cols
  ✓ Loaded promotions.csv: 50 rows × 10 cols
  ✓ Loaded shipments.csv: 566,067 rows × 4 cols
  ✓ Loaded returns.csv: 39,939 rows × 7 cols
  ✓ Loaded reviews.csv: 113,551 rows × 7 cols
  ✓ Loaded inventory.csv: 60,247 rows × 17 cols
  ✓ Loaded web_traffic.csv: 3,652 rows × 7 cols
  ✓ Loaded sales.csv: 3,833 rows × 3 cols


In [5]:
products.head(4)

,product_id,product_name,category,segment,size,color,price,cogs
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633158,11371.919278
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717299,8573.172954


In [7]:
reviews.head(5)

,review_id,order_id,product_id,customer_id,review_date,rating,review_title
0,REV-0000001,1,2400,58578,2012-07-24,5,Highly recommend
1,REV-0000002,3,396,58811,2012-08-03,5,Very satisfied
2,REV-0000003,10,1431,49101,2012-07-23,5,Great quality
3,REV-0000005,16,1668,41028,2012-08-05,5,Great quality
4,REV-0000006,17,2352,42030,2012-07-17,4,Good overall


In [9]:
returns.head(5)

,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount
0,RET-000001,2,609,2012-07-25,late_delivery,6,52458.01
1,RET-000002,32,1862,2012-07-16,wrong_size,2,5141.37
2,RET-000003,35,2359,2012-07-16,wrong_size,1,5315.95
3,RET-000004,47,1449,2012-07-11,wrong_size,4,6493.75
4,RET-000005,47,1450,2012-07-25,wrong_size,1,1740.76


In [14]:
#products with margin
dim_products = products.copy()
dim_products['gross_margin_pct'] = ((dim_products['price'] - dim_products['cogs']) / dim_products['price'] * 100).round(2)
dim_products['profit_per_unit']  = (dim_products['price'] - dim_products['cogs']).round(2)

#rating stat
avg_rating = reviews.groupby('product_id')['rating'].agg(['mean','count']).reset_index()
avg_rating.columns = ['product_id', 'avg_rating', 'review_count']
avg_rating['avg_rating'] = avg_rating['avg_rating'].round(2)

#Return stats per product
ret_stats = returns.groupby('product_id').agg(
    total_returns = ('return_id', 'count'),
    total_return_qty = ('return_quantity', 'sum'),
    total_refund = ('refund_amount', 'sum')
).reset_index()


dim_products = dim_products.merge(avg_rating, on='product_id', how='left')
dim_products = dim_products.merge(ret_stats, on='product_id', how='left')

dim_products[['review_count', 'total_returns', 'total_return_qty']] = \
    dim_products[['review_count', 'total_returns', 'total_return_qty']].fillna(0).astype(int)
dim_products['total_refund'] = dim_products['total_refund'].fillna(0)

save(dim_products, 'dim_products.csv')

  → Saved dim_products.csv: 2,412 rows × 15 cols (0.3 MB)


In [15]:
save(geography, 'dim_geography.csv')
save(promotions, 'dim_promotions.csv')

  → Saved dim_geography.csv: 39,948 rows × 4 cols (1.3 MB)
  → Saved dim_promotions.csv: 50 rows × 10 cols (0.0 MB)


### big order table


In [19]:
full_orders = order_items.copy()
#orders
full_orders = full_orders.merge(
     orders[['order_id', 'order_date', 'customer_id', 'zip',
            'order_status', 'payment_method', 'device_type', 'order_source']],
    on='order_id', how='left'
)
#products
full_orders = full_orders.merge(
    products[['product_id', 'product_name', 'category', 'segment', 'size', 'color', 'price', 'cogs']],
    on='product_id', how='left'
)

#payment
full_orders = full_orders.merge(
    payments[['order_id', 'payment_value', 'installments']],
    on='order_id', how='left'
)
#geo
full_orders = full_orders.merge(
    geography[['zip', 'city', 'region']].drop_duplicates(subset='zip'),
    on='zip', how='left'
)
#cost+rev of each orders insight
full_orders['line_revenue']       = (full_orders['quantity'] * full_orders['unit_price']).round(2)
full_orders['line_cost']          = (full_orders['quantity'] * full_orders['cogs']).round(2)
full_orders['line_gross_profit']  = (full_orders['line_revenue'] - full_orders['line_cost']).round(2)
full_orders['line_margin_pct']    = np.where(
    full_orders['line_revenue'] > 0,
    (full_orders['line_gross_profit'] / full_orders['line_revenue'] * 100).round(2),
    0
)
#promo + money that discount
full_orders['has_promo']          = (~full_orders['promo_id'].isna() & (full_orders['promo_id'] != '')).astype(int)
full_orders['has_double_promo']   = (~full_orders['promo_id_2'].isna() & (full_orders['promo_id_2'] != '')).astype(int)
full_orders['discount_pct']       = np.where(
    full_orders['line_revenue'] + full_orders['discount_amount'] > 0,
    (full_orders['discount_amount'] / (full_orders['line_revenue'] + full_orders['discount_amount']) * 100).round(2),
    0
)
#Date insight
full_orders['order_year']    = full_orders['order_date'].dt.year
full_orders['order_month']   = full_orders['order_date'].dt.month
full_orders['order_quarter'] = full_orders['order_date'].dt.quarter
full_orders['order_dow']     = full_orders['order_date'].dt.dayofweek  # 0=Mon
full_orders['order_dow_name']= full_orders['order_date'].dt.day_name()
full_orders['order_ym']      = full_orders['order_date'].dt.to_period('M').astype(str)

save(full_orders, 'dim_full_orders.csv')









  → Saved dim_full_orders.csv: 714,669 rows × 38 cols (183.2 MB)


In [18]:
returns.head(5)

,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount
0,RET-000001,2,609,2012-07-25,late_delivery,6,52458.01
1,RET-000002,32,1862,2012-07-16,wrong_size,2,5141.37
2,RET-000003,35,2359,2012-07-16,wrong_size,1,5315.95
3,RET-000004,47,1449,2012-07-11,wrong_size,4,6493.75
4,RET-000005,47,1450,2012-07-25,wrong_size,1,1740.76


In [20]:
fact_ret = returns.copy()
fact_ret = fact_ret.merge(
    products[['product_id', 'product_name', 'category', 'segment', 'size', 'color', 'price', 'cogs']],
    on='product_id', how='left'
)
fact_ret = fact_ret.merge(
    orders[['order_id', 'order_date', 'customer_id', 'order_source', 'device_type']],
    on='order_id', how='left'
)
# Time to return
fact_ret['days_to_return'] = (fact_ret['return_date'] - fact_ret['order_date']).dt.days
fact_ret['return_year']    = fact_ret['return_date'].dt.year
fact_ret['return_month']   = fact_ret['return_date'].dt.month
fact_ret['return_ym']      = fact_ret['return_date'].dt.to_period('M').astype(str)

save(fact_ret, 'dim_full_returns.csv')

  → Saved dim_full_returns.csv: 39,939 rows × 22 cols (7.3 MB)


### CUSTOMERS RFM


In [26]:
geography.count()

zip         39948
city        39948
region      39948
district    39948
dtype: int64

In [27]:
ref_date = orders['order_date'].max() + pd.Timedelta(days=1)
rfm = orders.merge(payments[['order_id', 'payment_value']], on='order_id', how='left')
rfm = rfm[rfm['order_status'] != 'cancelled']  # Exclude cancelled

rfm_agg = rfm.groupby('customer_id').agg(
    last_order_date   = ('order_date', 'max'),
    first_order_date  = ('order_date', 'min'),
    frequency         = ('order_id', 'nunique'),
    monetary          = ('payment_value', 'sum'),
    avg_order_value   = ('payment_value', 'mean'),
).reset_index()

rfm_agg['recency_days'] = (ref_date - rfm_agg['last_order_date']).dt.days
rfm_agg['tenure_days']  = (ref_date - rfm_agg['first_order_date']).dt.days
rfm_agg['monetary']     = rfm_agg['monetary'].round(2)
rfm_agg['avg_order_value'] = rfm_agg['avg_order_value'].round(2)

#  RFM Scores (quintiles 1-5)
rfm_agg['R_score'] = pd.qcut(rfm_agg['recency_days'], q=5, labels=[5,4,3,2,1]).astype(int)
rfm_agg['F_score'] = pd.qcut(rfm_agg['frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm_agg['M_score'] = pd.qcut(rfm_agg['monetary'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm_agg['RFM_score'] = rfm_agg['R_score'] * 100 + rfm_agg['F_score'] * 10 + rfm_agg['M_score']

def rfm_segment(row):
    r, f, m = row['R_score'], row['F_score'], row['M_score']
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif f >= 4 and m >= 3:
        return 'Loyal'
    elif r >= 4:
        return 'Potential'
    else:
        return 'At Risk'
    
rfm_agg['rfm_segment'] = rfm_agg.apply(rfm_segment, axis=1)
dim_cust = customers.merge(rfm_agg, on='customer_id', how='left')
dim_cust = dim_cust.merge(
    geography[['zip', 'region']].drop_duplicates(subset='zip'), #take only first non zip code
    on='zip', how='left'
)
# Signup year/month
dim_cust['signup_year']  = dim_cust['signup_date'].dt.year
dim_cust['signup_month'] = dim_cust['signup_date'].dt.month
dim_cust['signup_ym']    = dim_cust['signup_date'].dt.to_period('M').astype(str)

# Fill NAs for customers without orders
dim_cust['frequency']    = dim_cust['frequency'].fillna(0).astype(int)
dim_cust['monetary']     = dim_cust['monetary'].fillna(0)
dim_cust['rfm_segment']  = dim_cust['rfm_segment'].fillna('Never Purchased')

print(f"  Reference date for Recency: {ref_date.date()}")
print(dim_cust['rfm_segment'].value_counts().to_string())
save(dim_cust, 'dim_customers_rfm.csv')


  Reference date for Recency: 2023-01-01
rfm_segment
At Risk            42362
Never Purchased    33807
Champions          22579
Loyal              12259
Potential          10923
  → Saved dim_customers_rfm.csv: 121,930 rows × 23 cols (17.1 MB)


In [28]:
fact_ship = shipments.copy()
fact_ship['delivery_days']     = (fact_ship['delivery_date'] - fact_ship['ship_date']).dt.days
fact_ship['processing_days']   = None  # Will compute after join

# Join order info for processing time
fact_ship = fact_ship.merge(
    orders[['order_id', 'order_date', 'zip', 'order_status']],
    on='order_id', how='left'
)
fact_ship['processing_days'] = (fact_ship['ship_date'] - fact_ship['order_date']).dt.days
fact_ship['total_lead_time'] = (fact_ship['delivery_date'] - fact_ship['order_date']).dt.days

# Join geography for region
fact_ship = fact_ship.merge(
    geography[['zip', 'region', 'city']].drop_duplicates(subset='zip'),
    on='zip', how='left'
)

# Date dims
fact_ship['ship_year']  = fact_ship['ship_date'].dt.year
fact_ship['ship_month'] = fact_ship['ship_date'].dt.month
fact_ship['ship_ym']    = fact_ship['ship_date'].dt.to_period('M').astype(str)

# Delivery performance buckets
fact_ship['delivery_bucket'] = pd.cut(
    fact_ship['delivery_days'],
    bins=[-1, 2, 5, 7, 14, 999],
    labels=['1-2 days', '3-5 days', '6-7 days', '8-14 days', '15+ days']
)

save(fact_ship, 'dim_shipments_full.csv')

  → Saved dim_shipments_full.csv: 566,067 rows × 16 cols (57.7 MB)


In [29]:
fact_sales = sales.copy()
fact_sales.columns = ['date', 'revenue', 'cogs']
fact_sales['gross_profit']    = (fact_sales['revenue'] - fact_sales['cogs']).round(2)
fact_sales['gross_margin_pct']= (fact_sales['gross_profit'] / fact_sales['revenue'] * 100).round(2)

# Date dimensions
fact_sales['year']         = fact_sales['date'].dt.year
fact_sales['month']        = fact_sales['date'].dt.month
fact_sales['quarter']      = fact_sales['date'].dt.quarter
fact_sales['day_of_week']  = fact_sales['date'].dt.dayofweek
fact_sales['dow_name']     = fact_sales['date'].dt.day_name()
fact_sales['is_weekend']   = (fact_sales['day_of_week'] >= 5).astype(int)
fact_sales['year_month']   = fact_sales['date'].dt.to_period('M').astype(str)

# Rolling averages depend on chronological
fact_sales = fact_sales.sort_values('date')
fact_sales['revenue_ma7']  = fact_sales['revenue'].rolling(7, min_periods=1).mean().round(2)
fact_sales['revenue_ma30'] = fact_sales['revenue'].rolling(30, min_periods=1).mean().round(2)
fact_sales['cogs_ma7']     = fact_sales['cogs'].rolling(7, min_periods=1).mean().round(2)
fact_sales['cogs_ma30']    = fact_sales['cogs'].rolling(30, min_periods=1).mean().round(2)

# YoY growth (compare to same day last year)
fact_sales['revenue_ly'] = fact_sales['revenue'].shift(365)
fact_sales['yoy_growth_pct'] = np.where(
    fact_sales['revenue_ly'] > 0,
    ((fact_sales['revenue'] - fact_sales['revenue_ly']) / fact_sales['revenue_ly'] * 100).round(2),
    np.nan
)
fact_sales.drop(columns=['revenue_ly'], inplace=True)

save(fact_sales, 'dim_sales_full.csv')

  → Saved dim_sales_full.csv: 3,833 rows × 17 cols (0.5 MB)
